# SAADE–AUA 2026 v2 — повний експеримент і візуалізація аудіосценаріїв

**Окремий експеримент** на основі AUA. **Не перезаписує** SAADE v1 і вихідні файли AUA.

Дослідницькі блоки: (1) source-confounding та real/fake, (2) метадані мовців/генераторів, (3) 800–1500 *неперекривних* аудіосегментів, (4) E1/E2/E3 з 3–5 seeds, (5) справжні додаткові архітектури WavLM/AASIST за наявності залежностей, (6) ASRI, EER, ROC-AUC, bootstrap, GEE `model × regime`, (7) одноразовий hidden test. Окремо — **21 аудіосценарій, 21 аудіоплеєр, осцилограми, спектрограми, теплові карти, помилки та графіки кожного сценарію**.

Пілотний режим можна виконати навіть із джерелами, у яких `source` однозначно визначає real/fake. Для **науково підтвердженої незалежності від джерела** потрібна зіставна реальна та синтетична вибірка; лише кодом її не створити. `PAIRING_POLICY='strict_source'` блокує навчання без такого джерела.

Вхід: серверний каталог `/home/jovyan/work/AUA`. У папці з ноутбуком потрібні `saade_core.py`, `saade_visuals.py`, `saade_v2_methods.py`. GPU бажаний, Python 3.10+, `ffmpeg` із MP3-кодером.

## Етап 0. Налаштування та безпечний старт

Для першої перевірки використовуйте короткий запуск з трьома базовими моделями й 1–2 епохами. Для фінального дослідження — 4+ епохи, повтори по 3–5 seed і, за можливості, AASIST/WavLM. Ці зміни **вимагають нового вихідного каталогу**, якщо дані вже зафіксовані.

In [ ]:
from pathlib import Path
import sys, dataclasses, json, shutil, importlib.util, time
import numpy as np
import pandas as pd
from IPython.display import display, Audio, Markdown

HERE=Path.cwd()
if str(HERE) not in sys.path:sys.path.insert(0,str(HERE))
for filename in ('saade_core.py','saade_visuals.py','saade_v2_methods.py'):
    if not (HERE/filename).is_file():
        raise FileNotFoundError(f'Потрібен модуль {filename} поруч із ноутбуком. Розпакуйте повний ZIP.')
import saade_core as S
import saade_visuals as V
import saade_v2_methods as M

DATA_ROOT=Path('/home/jovyan/work/AUA')
EXP_NAME='SAADE_AUA_2026_v2_5level'
OUTPUT=Path('/home/jovyan/work/results')/EXP_NAME
# Строгий режим доступний тільки з надійними real/fake в одному джерелі.
PAIRING_POLICY='exploratory'   # 'strict_source' — обов'язково для сильних висновків про source robustness
SCENARIO_PROFILE='v2_5level'  # 21 сценарій; або 'v1_3level' для історичного зіставлення
RUN_WAVLM=False              # True після встановлення transformers + завантаження checkpoint
RUN_AASIST=False             # True після клонування оригінального репозиторію AASIST
RUN_RAWNET2=False            # True після клонування оригінального репозиторію ASVspoof5
RUN_SEED_REPEATS=False       # True: 3 повні додаткові повтори (витратний GPU-експеримент)
RUN_FINAL_HIDDEN_TEST=False  # True тільки ОДИН РАЗ після остаточного затвердження протоколу
REQUIRE_MATCHED_HIDDEN=True  # заборонити фінальний тест, якщо джерела real/fake повністю розділені
SEEDS=(42,123,777)          # від 3 до 5 різних seed
TARGET_SEGMENTS=1200        # ціль — 800–1500; фактична кількість може бути меншою
MAX_BASE_FILES=500

model_names=['spectrogram_cnn','raw_wave_cnn','mfcc_svm']
if RUN_WAVLM:model_names.append('wavlm')
if RUN_AASIST:model_names.append('aasist')
if RUN_RAWNET2:model_names.append('rawnet2')
cfg=S.Config(root=str(DATA_ROOT),out=str(OUTPUT),seed=SEEDS[0],scenario_seed=1001,
    scenario_profile=SCENARIO_PROFILE,seconds=4.0,sample_rate=16000,
    max_items=MAX_BASE_FILES,max_per_group=30,epochs=4,batch_size=12,
    models=tuple(model_names),bootstraps=500,allow_weak_groups=False)
OUTPUT.mkdir(parents=True,exist_ok=True)
assert (DATA_ROOT/'01_audio').is_dir(), f'Немає {DATA_ROOT}/01_audio'
assert shutil.which('ffmpeg'), 'Потрібно встановити ffmpeg'
print('AUA:',DATA_ROOT,'| OUTPUT:',OUTPUT,'| scenarios:',1+4*len(S.available_levels(cfg.scenario_profile)))
print('Models:',cfg.models,'| repeats:',RUN_SEED_REPEATS,'| matched-policy:',PAIRING_POLICY)
print('CUDA:')
import torch
print(torch.__version__,torch.cuda.is_available(),torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

### 0.1. Бібліотеки

За потреби виконайте `%pip install soundfile statsmodels ipywidgets` і перезапустіть kernel. Для WavLM додатково встановіть відповідну до вашого PyTorch версію `transformers`. Не встановлюйте PyTorch наново без потреби — це може зламати CUDA.

In [ ]:
import importlib.util
names=['numpy','pandas','scipy','soundfile','sklearn','joblib','matplotlib','statsmodels','torch']
missing=[n for n in names if importlib.util.find_spec(n) is None]
print('Відсутні бібліотеки:',missing or 'немає')
if missing:raise RuntimeError('Спершу встановіть відсутні пакети в kernel JupyterLab')
if RUN_WAVLM and importlib.util.find_spec('transformers') is None:
    raise RuntimeError('WavLM потребує transformers: %pip install transformers')
if RUN_AASIST and not (Path(cfg.repo_aasist)/'models'/'AASIST.py').exists():
    raise RuntimeError('Клонуйте офіційний https://github.com/clovaai/aasist')
if RUN_RAWNET2 and not (Path(cfg.repo_asvspoof5)/'Baseline-RawNet2'/'model.py').exists():
    raise RuntimeError('Клонуйте офіційний https://github.com/asvspoof-challenge/asvspoof5')
print('Середовище пройшло перевірку')

## Етап 1. Каталог, ліцензії, мітки, походження

Автоматично зчитуємо AUA, підраховуємо реальні файли один раз. Не зараховуємо службові копії `05_splits` як нові записи. Дані з неперевіреними групами **не допускаються** до незалежного розбиття.

In [ ]:
catalog=S.build_catalog(cfg)
source_all,matched_texts,quality=M.source_balance_audit(catalog,cfg,'all')
print('Усього:',len(catalog),'| перевірені мітки:',catalog.label.notna().sum())
display(source_all)
print('Зіставні джерела (обидва класи):',quality['matched_sources'])
print('Відповідні транскрипти real/fake:',quality['matched_text_groups'])
print('ПОПЕРЕДЖЕННЯ:',quality['warnings'])
display(catalog.groupby(['source','label','group_confidence'],dropna=False).size().reset_index(name='n').head(30))

## Етап 2. Зіставна вибірка та незалежні первинні split

**Застереження:** однаковий corpus/source для двох класів — лише *необхідна* умова зіставності, а не підтвердження однакових мікрофонів, записів чи мовців. У строгому режимі без обох класів в одному джерелі код зупиниться. Файли із `weak_file_fallback` виключаються. Розбиття створюється **до** нарізання на сегменти.

In [ ]:
parents,source_diagnostic=M.check_and_prepare_parent_split(
    catalog,cfg,max_items=MAX_BASE_FILES,policy=PAIRING_POLICY)
print('ПЕРВИННІ ЗАПИСИ:',len(parents),'| незалежних груп:',parents.group_id.nunique())
display(parents.groupby(['split','label','source']).agg(files=('path','size'),groups=('group_id','nunique')).reset_index())
assert not (parents.groupby('group_id').split.nunique()>1).any(), 'Витік груп!' 

## Етап 3. 800–1500 неперекривних аудіофрагментів

З довгих записів створюємо 4-секундні вікна, що **не перекриваються**. Вони успадковують `group_id` та `split` первинного аудіо. Це збільшує кількість тестових фрагментів, але **не кількість незалежних мовців**. Коли даних замало, код чесно показує дефіцит.

In [ ]:
splits,segment_counts,segment_info=M.expand_nonoverlapping_segments(
    parents,cfg,target=TARGET_SEGMENTS,max_per_parent=30)
print('SEGMENT REPORT:',segment_info)
display(segment_counts)
for split_name in ('train','validation','hidden_test'):
    xx=splits[splits.split.eq(split_name)]
    assert xx.label.nunique()==2, f'У {split_name} немає обох класів'
S.lock_hidden_test(splits,cfg)
generator_table,generator_report=M.generator_independence(splits,cfg)
print('Перевірка генераторів:',generator_report)
display(generator_table.head(40))
display(M.final_technical_summary(cfg,splits,source_report=quality,segment_report=segment_info))

## Етап 4. **Усі 21 сценарій:** генерація, ілюстрації, прослуховування

Для одного REAL і одного FAKE з validation створюються всі сценарії. **Кожен з 21 аудіофрагмента можна відтворити**. Формуються великі панелі з осцилограмами й спектрограмами, а також CSV з параметрами, RMS і кореляцією. Файли AUA не змінюються.

П'ять рівнів нового протоколу впорядковані за інтенсивністю: Noise 20/15/10/5/0 dB SNR; MP3 64/48/32/24/16 kbps; Reverberation RT60 0,2/0,35/0,5/0,7/0,9 c; Low-pass 7000/6000/5000/4200/3400 Hz.

In [ ]:
guide=V.draw_scenario_guide(cfg,tag='validation')
examples=V.pick_examples(splits,'validation')
for record in examples:
    print('\n####', 'SYNTHETIC' if int(record.label)==1 else 'REAL')
    audio_stats=V.show_audio_gallery(record,cfg,show_players=True,show_figures=True,tag='validation')
print('Графіки збережено:',OUTPUT/'visuals'/'validation')

### 4.1. Інтерактивний вибір аудіозапису

Можна вибрати будь-який validation-запис, категорію й рівень спотворення та прослухати результат. `ipywidgets` є необов'язковим: попередня комірка вже показує всі плеєри.

In [ ]:
interactive_player=V.make_interactive_player(splits,cfg,'validation')

## Етап 5. E1 — навчання без аугментацій (Static)

Три базові архітектури: Spectrogram CNN, Raw Wave CNN, MFCC-SVM. **WavLM, AASIST і RawNet2** можуть бути підключені через `RUN_WAVLM`, `RUN_AASIST`, `RUN_RAWNET2` і використовують справжні реалізації/ваги, а не перейменовані CNN. Після будь-якої зміни складу моделей починайте **новий** експериментальний каталог.

In [ ]:
models={};infos={};base_parts=[]
for name in cfg.models:
    print('\nE1 Static:',name)
    model,meta=S.train_one(splits,cfg,name,regime='static')
    models[(name,'static')]=model;infos[(name,'static')]=meta
    pred=S.run_scenario_predictions(splits,model,name,'static',cfg,
          threshold=meta['threshold'],split='validation')
    base_parts.append(pred)
base_val=pd.concat(base_parts,ignore_index=True)
print('E1 validation rows:',len(base_val))

## Етап 6. Адаптивний вибір категорій (без hidden test)

Ваги E3 розраховуються лише за результатами E1 на **validation**. Змінюється частота аугментації категорій, а не первинна мітка real/fake. Використання validation для визначення ваг робить валідаційне порівняння *дослідницьким*, а не незалежним підтвердженням.

In [ ]:
weights_by_model={name:S.validation_difficulty(base_val[base_val.model.eq(name)]) for name in cfg.models}
weights={c:float(np.mean([weights_by_model[n][c] for n in cfg.models])) for c in S.CATEGORIES}
wtotal=sum(weights.values())
weights={k:v/wtotal for k,v in weights.items()}
S.save_json(OUTPUT/'05_adaptive_weights.json',weights)
print('Адаптивні ваги:',weights)
display(pd.DataFrame([{'category':k,'selection_probability':v} for k,v in weights.items()]))

## Етап 7. E2 (Random) і E3 (Adaptive)

Однакова кількість епох, однакова базова train/validation вибірка, однакові набори сценаріїв. Кожна модель навчається заново для кожного режиму. Фіксований `scenario_seed` робить контрольні шумові варіанти однаковими під час порівняння seeds.

In [ ]:
parts=[base_val]
for name in cfg.models:
    for regime in ('random','adaptive'):
        print('\n',name,regime)
        model,meta=S.train_one(splits,cfg,name,regime=regime,
             adaptive_weights=(weights if regime=='adaptive' else None))
        models[(name,regime)]=model;infos[(name,regime)]=meta
        p=S.run_scenario_predictions(splits,model,name,regime,cfg,
                  threshold=meta['threshold'],split='validation')
        parts.append(p)
validation_predictions=pd.concat(parts,ignore_index=True)
validation_predictions.to_csv(OUTPUT/'04_val_predictions_ALL.csv',index=False)
print('Усього сценарних рішень:',len(validation_predictions))

## Етап 8. **Тестування ВІДНОСНО КОЖНОГО СЦЕНАРІЮ**

Таблиці й графіки для всіх **21 сценарію × 3 режими × N моделей**: Balanced Accuracy, EER, ROC-AUC, F1, FAR, miss rate, зміни від baseline, ASRI, heatmaps, криві за рівнями, confusion matrices і приклади помилкових рішень. Самі значення візьмуться з реального запуску, не з готових демонстрацій.

In [ ]:
val_metrics,val_asri,coverage=V.report_scenario_tests(
    validation_predictions,cfg,tag='validation',show=True)
display(val_asri.sort_values(['model','regime']))
V.plot_roc_for_selected(validation_predictions,cfg,model='spectrogram_cnn',regime='adaptive',tag='validation')
for record in examples:
    V.show_one_record_decisions(validation_predictions,record,cfg,tag='validation')

## Етап 9. Статистика та контроль змішування джерел

GEE із взаємодіями `model × regime` і `category × level` (якщо достатньо незалежних груп), парний cluster-bootstrap за групами, таблиці ASRI. Якщо джерело однозначно визначає клас, така статистика сама собою не усуває конфаундинг.

In [ ]:
seed0_stats=M.interaction_statistics(validation_predictions,cfg,'validation')
print('Статистика взаємодій:',seed0_stats)
for model in cfg.models:
    try:
        boot=S.paired_group_bootstrap_delta(validation_predictions,model,B=cfg.bootstraps,seed=cfg.seed)
        print(model,boot)
    except Exception as exc:
        print('Bootstrap CHECK:',model,str(exc)[:220])
strata=S.summarize_source_results(validation_predictions,cfg)
display(strata.head(30))
source_performance=M.source_balance_audit(splits,cfg,'selected')[2]
display(M.final_technical_summary(cfg,splits,validation_predictions,source_performance,segment_info))

## Етап 10. Повтори на **3–5 різних seeds**

Для статті потрібно увімкнути `RUN_SEED_REPEATS=True` та завершити всі повтори. Усі seeds використовують **той самий зафіксований split**, але навчають моделі заново. Приблизний час залежить від GPU/CPU. Повтори мають окремі вихідні підкаталоги; дані AUA залишаються недоторканними.

In [ ]:
if RUN_SEED_REPEATS:
    if len(SEEDS)<3 or len(SEEDS)>5:raise ValueError('Потрібно 3–5 seeds')
    replicates,replicate_summary,all_seed_predictions=M.seed_replicates(
        cfg,splits,seeds=SEEDS,models=cfg.models)
    print('Статистика ASRI між seeds:')
    display(replicate_summary)
else:
    print('Повтори не запускалися. Для завершення вимоги #4 встановіть RUN_SEED_REPEATS=True та виконайте цю клітинку.')

## Етап 11. Фіксація експерименту **ДО** hidden test

Перевіряємо, що набори, checkpoints та пороги збережені. Будь-які зміни параметрів або вибірки після фіксації означають **новий ідентифікатор експерименту**, а не правку результатів v2.

In [ ]:
frozen=S.save_locked_protocol(cfg,splits,infos)
print('Фіксований SHA розбиття:',frozen['dataset_hash'])
print('Кількість моделей × режимів:',len(frozen['models']))
if RUN_SEED_REPEATS:
    print('Replicates:',list(SEEDS))
else:print('CHECK: публікаційні 3–5 seeds ще не завершено')

## Етап 12. ФІНАЛЬНИЙ hidden test — **один запуск**

За замовчуванням `False`. Для незалежного підтвердження потрібні: зафіксовані checkpoints, виконані повтори, source-matching (якщо `REQUIRE_MATCHED_HIDDEN=True`), **відсутність попереднього hidden запуску**. Після активації створюється маркер одноразового тесту — навіть якщо обчислення перерветься, його не можна потайки повторити.

In [ ]:
if RUN_FINAL_HIDDEN_TEST:
    if REQUIRE_MATCHED_HIDDEN and not quality['matched_sources']:
        raise RuntimeError('HIDDEN TEST BLOCKED: real/fake незіставні за джерелами. Зберіть коректні пари та почніть новий експеримент.')
    if not RUN_SEED_REPEATS:
        raise RuntimeError('HIDDEN TEST BLOCKED: не виконано 3–5 незалежних seeds.')
    cfg.test_finalization=True
    S.ensure_test_gate(cfg)
    S.verify_locked_protocol(cfg,splits)
    marker=M.reserve_hidden_evaluation(cfg,splits)
    # ОДИН протокол hidden, усі зафіксовані моделі та seed — окремими повтореннями.
    hidden_all,hidden_by_seed,hidden_summary=M.hidden_all_repeats(cfg,splits,SEEDS,cfg.models)
    print('FINAL HIDDEN ASRI — mean/std across training seeds:')
    display(hidden_summary)
    # Розподіл сценаріїв без псевдореплікації seeds: показуємо кожен seed окремо.
    for selected_seed in SEEDS:
        sub=hidden_all[hidden_all.training_seed.eq(selected_seed)]
        hm,ha,hc=V.report_scenario_tests(sub,cfg,tag=f'hidden_seed_{selected_seed}',show=True)
        M.interaction_statistics(sub,cfg,tag=f'hidden_seed_{selected_seed}')
else:
    print('Hidden test залишається закритим. Зараз доступні лише дослідницькі validation-результати.')

## Етап 13. Інтерпретація та експорт

Звіт містить усі CSV/JSON/PNG, журнали моделей та протокол. За замовчуванням **не включає** аудіосигнали та персональні дані. Для прослуховування використовуйте цей ноутбук або окремий Viewer. Самі метрики не підтверджують справедливість/генераторну незалежність, якщо для цього немає відповідних даних.

In [ ]:
print(S.make_readme(cfg,splits))
archive=S.zip_results(cfg,include_models=False)
print('Архів звітів:',archive)
print('Нотатка: це окремий SAADE v2; не перезаписуйте v1.')

---
## Контрольний список перед статтею

- [ ] Відібрані зіставні REAL/FAKE приклади, тобто джерело запису не є простим ідентифікатором класу.
- [ ] Підтверджені speaker/source_record/generator/voice ID, немає групових витоків.
- [ ] У звіті чесно вказано фактичні 800–1500 сегментів і **окремо** кількість незалежних груп.
- [ ] Завершено 3–5 seeds для всіх архітектур і режимів.
- [ ] AASIST/WavLM перевірено, якщо вони включені до висновків (необов'язкові до стартового запуску).
- [ ] Всі 21 сценарій мають BA, EER, AUC, F1, FAR, Miss, ASRI та візуалізації.
- [ ] Hidden test виконаний **один раз**, після фіксації протоколу.

**Наукове застереження:** якщо зіставні real/fake у наявному AUA відсутні, перший запуск v2 залишається пілотним, незалежно від обсягів чи гарних графіків.